### Part 1
Fetch RSS Feed into the context

In [ ]:
import requests

In [33]:
ADVERTS_RSS_URL = "https://www.adverts.ie/for-sale/rss/$"

In [34]:
def fetch_rss_feed(url: str) -> str:
  """Fetch raw XML/RSS content with a descriptive User-Agent and timeout."""
  headers = {
      "User-Agent": (
          "AcademicDataPipeline/1.0 (Student AI Project; contact via 20101674@mydbs.ie)"
      ),
      "Accept": "application/rss+xml, application/xml, text/xml;q=0.9, */*;q=0.8",
  }

  response = requests.get(url, headers=headers, timeout=15)
  response.raise_for_status()  # Raises HTTPError if occured
  return response.text

In [35]:
raw_xml = fetch_rss_feed(ADVERTS_RSS_URL)
raw_xml

'<?xml version="1.0" encoding="utf-8"?>\n\n<rss version="2.0">\n\t<channel>\n\t<title>Adverts.ie - Recent Ads All 971,690 listings</title>\n\t<link>https://www.adverts.ie/for-sale/</link>\n\t<description>Adverts.ie - Recent Ads All 971,690 listings</description>\n\t<language>en</language>\n\n<item>\n\t<title>Samsung 8GB DDR4 3200 MHz SODIMM Laptop Memory</title>\n\t\t<price><![CDATA[40]]></price>\n\t\t\t<county><![CDATA[Dublin]]></county>\n\t\t\t<area><![CDATA[Ballsbridge]]></area>\n\t\t\t<image_url>https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiYjU0NzZlZjVkNjBkM2RkYzUyMmNkMGViYWFmNmVkOTM2NjU1MzljYTVlMWI3ZTM2ODM1NzE3NjM4MWNjNGFkOC5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=448b308f15ce119a00b5d1e8ec015b99fdb11bcd06b92a5b4cc824f3c3b1af3c</image_url>\n\t\t<link>https://www.adverts.ie/laptop-parts-and-accessories/samsung-8gb-ddr4-3200-mhz-sodimm-laptop-memory/40749738</link>\t\n\t<guid>40749738</gui

    Item data-structure in XML Response is as below :-

        <item>\n\t
            <title>{{Title}}</title>\n\t\t
            <price><![CDATA[price]]></price>\n\t\t\t
            <county><![CDATA[county]]></county>\n\t\t\t
            <area><![CDATA[area]]></area>\n\t\t\t
            <image_url>{{Image}}</image_url>\n\t\t
            <link>{{Link}}</link>\t\n\t
            <guid>{{unique number}}</guid>\n\t
            <pubDate>{{Date}}</pubDate>\n\t
            <description>{{TEXT}}</description>\n
        </item>\n\n
    
    Note:-
    In XML, <![CDATA[SAMPLE]]> is a CDATA (Character Data) section that tells the XML parser to treat the word "SAMPLE" strictly as plain, literal text rather than markup.

### Part 2
Store the raw RSS feed into designated file and then parse the XML Response.

Folder structure for storing feed: YEAR -> MONTH -> DAY -> NOW.xml (One file per xml response is stored in day, month and year wise manner)

In [36]:
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
import xml.etree.ElementTree as ET

In [37]:
RAW_DIR = Path("data/raw")

In [38]:
def save_raw_xml(xml_bytes: bytes, base_dir: Path = RAW_DIR) -> Path:
    """Save the response as data/raw/<year>/<month>/<day>/<timestamp>.xml"""
    now = datetime.now(timezone.utc)
    folder = base_dir / f"{now:%Y}" / f"{now:%m}" / f"{now:%d}"
    folder.mkdir(parents=True, exist_ok=True)

    file_path = folder / f"{now:%Y-%m-%d_%H%M%S}.xml"
    file_path.write_bytes(xml_bytes)
    return file_path


In [39]:
# AI Assisted Functions below :-
def get_text(item, tag):
    """Return the text inside a tag, or None if it is missing or empty."""
    element = item.find(tag)
    if element is not None and element.text:
        return element.text.strip()
    return None


def parse_date(value):
    """Convert an RSS date string into a datetime."""
    if value is None:
        return None
    return parsedate_to_datetime(value)


def parse_rss(xml_bytes: bytes) -> list[dict]:
    """Turn the RSS XML into a list of listing dictionaries."""
    root = ET.fromstring(xml_bytes)
    listings = []

    for item in root.iter("item"):
        listings.append({
            "area": get_text(item, "area"),
            "guid": get_text(item, "guid"),
            "price": get_text(item, "price"),
            "title": get_text(item, "title"),
            "county": get_text(item, "county"),
            "category": get_text(item, "category"),
            "image_url": get_text(item, "image_url"),
            "description": get_text(item, "description"),
            "published_at": parse_date(get_text(item, "pubDate")),
        })
    return listings

In [40]:
saved_path = save_raw_xml(raw_xml.encode("utf-8")) # converts into bytes
print(saved_path)

data\raw\2026\11\26\2026-11-26_231134.xml


In [41]:
listings = parse_rss(raw_xml)
listings

[{'area': 'Ballsbridge',
  'guid': '40749738',
  'price': '40',
  'title': 'Samsung 8GB DDR4 3200 MHz SODIMM Laptop Memory',
  'county': 'Dublin',
  'category': None,
  'image_url': 'https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiYjU0NzZlZjVkNjBkM2RkYzUyMmNkMGViYWFmNmVkOTM2NjU1MzljYTVlMWI3ZTM2ODM1NzE3NjM4MWNjNGFkOC5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=448b308f15ce119a00b5d1e8ec015b99fdb11bcd06b92a5b4cc824f3c3b1af3c',
  'description': '<div>\n\t\t\t\t<div>\n\t\t\t\t\t\t\t\t\t\t<a href="https://www.adverts.ie/laptop-parts-and-accessories/samsung-8gb-ddr4-3200-mhz-sodimm-laptop-memory/40749738">\n\t\t\t\t\t\t\t<img src="https://media.adverts.ie/eyJidWNrZXQiOiJtZWRpYS5hZHNpbWcuY29tIiwia2V5IjoiYjU0NzZlZjVkNjBkM2RkYzUyMmNkMGViYWFmNmVkOTM2NjU1MzljYTVlMWI3ZTM2ODM1NzE3NjM4MWNjNGFkOC5qcGciLCJvdXRwdXRGb3JtYXQiOiJqcGVnIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoyMjcsImhlaWdodCI6MTg3fX19?signature=448b308f15